# SampleNTT: predict rejection from one power trace

**Husky + CW308_STM32F4 · variable-time sampler A0**

This lab demonstrates a narrow, testable leakage claim: a classifier can predict whether the **first 12-bit candidate is accepted** from one power capture of an unseen public seed. Train on known seeds, freeze the classifier, then test on different seeds.

The capture window was fixed using the previously validated experiment. Both classes have the same total sampling workload. This demonstrates local accept/reject leakage; it does not recover an ML-KEM key.

Start a fresh kernel using chipwhisperer-new/.venv. Run the seven code cells in order. Hardware routines, raw records and metadata stay in the companion directory.

## 1. Setup

Use 128 seeds per class for training and another 128 per class for testing: **512 captures, one per seed**. Gain starts at the previously usable 10 dB. To resume an interrupted run, set RESUME_RUN to its printed results folder; completed captures and the frozen model are reused.

In [ ]:
from pathlib import Path
import sys, importlib
REPO = Path.home() / 'Desktop/work/chipwhisperer-new'
LAB = REPO / 'jupyter/courses/custom/mlkem-samplentt-power-lab'
sys.path.insert(0, str(LAB))
import capture.lab as lab_workflow
importlib.reload(lab_workflow)

PER_CLASS = 128
GAIN_DB = 10.0
RESUME_RUN = None
lab = lab_workflow.SampleNTTPowerLab(REPO, per_class=PER_CLASS,
                                    gain_db=GAIN_DB, resume=RESUME_RUN)

## 2. Prepare and verify the hardware

This builds and programs the **STM32F4 target**, connects Husky, verifies the sampler identity and polynomial digest, and warms the target before collecting data. It also restores normal USB–FPGA bus timing.

The fixed 512-sample window covers about 17.4 µs near the first candidate pair, at four ADC samples per CPU cycle. It is tied to the calibrated firmware image; a different image stops the lab and requires recalibration. Clipping and capture errors remain enabled.

In [ ]:
check = lab.prepare()

## 3. Capture training data

Seeds are balanced between first-candidate acceptance and rejection. Every seed takes exactly 160 pairs, with a valid second candidate and output completion on the final candidate. Trial order is randomized; every successful capture is checkpointed and verified. Failures stop capture rather than silently dropping seeds. Rerun this cell to resume after fixing an error.

In [ ]:
lab.collect_training();

## 4. Freeze the distinguisher

The classifier selects 16 power-sample positions and fits a regularized linear model using **training data only**. The window was fixed before this run. The model, threshold and capture settings are saved before test capture. A separate total-duration classifier provides a timing baseline.

In [ ]:
lab.freeze();

## 5. Capture fresh test seeds

These seeds have not appeared in training or earlier saved experiments. The capture code requires the frozen model and unchanged settings. Do not return to model or window selection after inspecting this test.

In [ ]:
lab.collect_test();

## 6. Evaluate the claim

**Balanced accuracy** averages accept and reject recall; chance is 50%. **AUC** measures how well the scores separate the classes. Confidence bounds account for finite test seeds even if every prediction is correct. The permutation p-value tests label association with the already frozen predictions.

A convincing run has a power accuracy interval above 50% and a small p-value. Compare against total timing and inspect the waveform/score plots. A negative result is retained. Results apply to this balanced seed population, warmed target and exact build; fresh-session replication strengthens the claim.

In [ ]:
report = lab.results()

## 7. Disconnect

Release the board for another notebook.

In [ ]:
lab.close()

The [full calibration and experimental method](mlkem-samplentt-power-lab/docs/Full%20SampleNTT%20Method.ipynb), original measured run, and Git checkpoint remain available. Source integrity checks, per-trace digests, raw waveforms, failure logs and model fingerprints are retained by the helper code.